# 04b · Modèle 2 : risque de retard, **tous les modèles de classification**
**Projet : E-Commerce Intelligence Platform** · CRISP-DM, phase 4 (Modeling) et phase 5 (Evaluation)

**Objectif** : estimer, dès l'achat, la probabilité qu'une commande soit livrée **après la date promise** (`is_late = 1`), pour déclencher des actions préventives (alerte vendeur, message client, suivi prioritaire).

**Particularités** : classes **déséquilibrées** (environ 8 % de retards), taux de retard **instable** d'un bloc à l'autre (7,7 % train, 13,5 % valid, 6,2 % test). Métriques : **PR-AUC** (comparée au taux de base), rappel, précision, coût d'une décision.

**Démarche**
1. Baselines par règles simples (historique du vendeur, du trajet, promesse courte).
2. **Tous les modèles de classification** comparés sur valid : Régression logistique, Ridge, LDA, Naive Bayes, KNN, SVM (RBF et linéaire), MLP, Decision Tree, Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, AdaBoost, XGBoost, LightGBM, CatBoost.
3. Réglage des 3 meilleurs boosters (validation croisée **temporelle**, PR-AUC), puis **mélange** de leurs probabilités.
4. Seuil de décision choisi sur **valid** (coût, précision cible, capacité), évalué sur **test**.
5. Mini-expérience : classification directe contre « régression puis seuil ».

> Données : datasets Kaggle `e-commerce-dataset-model-2` (et `e-commerce-dataset-model-1` pour la mini-expérience, qui a besoin de `delivery_days`). Durée indicative : 15 à 30 minutes.

## 1. Imports et paramètres

In [ ]:
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import lightgbm as lgb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, AdaBoostClassifier)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve, brier_score_loss)
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
try:
    import xgboost as xgb; HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    from catboost import CatBoostClassifier; HAS_CAT = True
except Exception:
    HAS_CAT = False
print("XGBoost :", HAS_XGB, "| CatBoost :", HAS_CAT)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEED = 42
rng = np.random.default_rng(SEED)
FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/models")
os.makedirs(OUT_DIR, exist_ok=True)

USE_PROMISE = None             # None : choix automatique (PR-AUC sur valid) ; True ou False pour forcer
SKIP_MODELS = set()            # ex. {"SVM (rbf)", "KNN"} pour gagner du temps
N_EST    = 120 if FAST else 400
RF_TREES = 30 if FAST else 150
GB_TREES = 40 if FAST else 120
SLOW_SUB = 1500 if FAST else 12000
N_ITER_SEARCH = 2 if FAST else 8

# Hypothèses de coût (à adapter au métier) : un retard non détecté coûte COST_FN, une fausse alerte COST_FP
COST_FN, COST_FP = 5.0, 1.0
PRECISION_TARGET = 0.30
TOPK_SHARE = 0.10              # capacité d'action : 10 % des commandes peuvent être traitées

# Chemins des jeux de données Kaggle (créés à partir de la sortie du notebook 03)
DATA_PATHS = {
    "model1_delivery_duration.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-1/model1_delivery_duration.csv",
    "model2_late_risk.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-2/model2_late_risk.csv",
    "model3_customer_segmentation.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-3/model3_customer_segmentation.csv",
}

def find_file(name):
    """DATA_DIR (si défini), puis le chemin Kaggle ci-dessus, puis une recherche dans /kaggle/input."""
    cands = []
    if os.environ.get("DATA_DIR"):
        cands.append(os.path.join(os.environ["DATA_DIR"], name))
    cands.append(DATA_PATHS.get(name, ""))
    cands += glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError(f"{name} introuvable : ajoute le dataset correspondant en entrée du notebook (Add Input).")

## 2. Chargement et blocs temporels

In [ ]:
m2 = pd.read_csv(find_file("model2_late_risk.csv"), parse_dates=["order_purchase_timestamp"])
try:     # delivery_days sert uniquement à la mini-expérience (jamais comme variable explicative)
    m1y = pd.read_csv(find_file("model1_delivery_duration.csv"), usecols=["order_id", "delivery_days"])
    m2 = m2.merge(m1y, on="order_id", how="left"); HAVE_M1 = True
except FileNotFoundError:
    HAVE_M1 = False
    print("Dataset du modèle 1 absent : la mini-expérience (cellule 15) sera ignorée.")

FEATURES_NUMERIC = [
    "nb_items", "multi_seller", "total_price", "total_freight", "total_weight_g", "total_volume_cm3",
    "weight_imputed", "freight_ratio", "limit_days", "distance_km", "distance_imputed", "same_state",
    "customer_lat", "customer_lng", "seller_lat", "seller_lng", "payment_installments",
    "purchase_month", "purchase_dow", "purchase_hour", "is_weekend", "is_black_friday_week", "orders_last_7d",
    "seller_n_past", "seller_mean_days", "seller_late_rate", "route_n_past", "route_mean_days", "route_late_rate",
    "log_total_price", "log_total_freight", "log_total_weight_g", "log_total_volume_cm3", "log_distance_km",
]
CATS = ["customer_state", "seller_state", "categorie_grouped", "payment_type"]
PROMISE = ["promised_days"]
FEATURES_CORE = FEATURES_NUMERIC + CATS

assert not [c for c in FEATURES_CORE + PROMISE + ["is_late", "split", "order_id"] if c not in m2.columns], "Colonnes absentes"

tr, va, te = [m2[m2["split"] == s].sort_values("order_purchase_timestamp").reset_index(drop=True)
              for s in ["train", "valid", "test"]]
ytr, yva, yte = tr["is_late"].values, va["is_late"].values, te["is_late"].values
display(pd.DataFrame({"commandes": [len(tr), len(va), len(te)], "retards": [ytr.sum(), yva.sum(), yte.sum()],
                      "taux_retard": [ytr.mean(), yva.mean(), yte.mean()]}, index=["train", "valid", "test"]))
RATIO = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
print(f"Rapport négatifs / positifs (train) : {RATIO:.1f}")

## 3. Baselines par règles simples (valid)
Taux de base (classement aléatoire), historique du vendeur, historique du trajet, promesse courte, distance / promesse. Un classifieur n'a de valeur que s'il bat ces règles.

In [ ]:
def pr_auc(y, s): return float(average_precision_score(y, s))
def roc_auc(y, s): return float(roc_auc_score(y, s))
def f1_max(y, s):
    p, r, _ = precision_recall_curve(y, s)
    return float(np.max(2 * p * r / np.maximum(p + r, 1e-12)))
def top_stats(y, s, share):
    k = max(int(len(y) * share), 1); idx = np.argsort(-s)[:k]
    return float(y[idx].mean()), float(y[idx].sum() / max(y.sum(), 1))          # précision, rappel

def clf_metrics(y, s, proba=True):
    prec, rec = top_stats(y, s, TOPK_SHARE)
    out = {"PR-AUC": pr_auc(y, s), "ROC-AUC": roc_auc(y, s), "F1 max": f1_max(y, s),
           "précision top 10 %": prec, "rappel top 10 %": rec}
    out["Brier"] = brier_score_loss(y, s) if proba and 0 <= np.min(s) and np.max(s) <= 1 else np.nan
    return out

base_rows = [{"modèle": "Taux de base (aléatoire)", "famille": "Baseline", "PR-AUC": float(yva.mean()), "ROC-AUC": 0.5}]
for n, s in [("Historique du vendeur", va["seller_late_rate"]), ("Historique du trajet", va["route_late_rate"]),
             ("Promesse courte", -va["promised_days"]), ("Distance / promesse", va["distance_km"] / va["promised_days"])]:
    base_rows.append({"modèle": n, "famille": "Baseline", **clf_metrics(yva, s.values, proba=False)})
display(pd.DataFrame(base_rows).set_index("modèle"))

## 4. Préparation des variables pour chaque famille de modèles

In [ ]:
class Clipper(BaseEstimator, TransformerMixin):
    """Plafonne chaque variable aux quantiles q et 1-q du jeu d'ajustement (limite l'effet des extrêmes)."""
    def __init__(self, q=0.001): self.q = q
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_, self.hi_ = np.nanquantile(X, self.q, axis=0), np.nanquantile(X, 1 - self.q, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def take(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]

def make_converter(feats, ref):
    """Catégories au format pandas « category » (LightGBM), niveaux fixés sur le jeu de référence."""
    cats = [c for c in CATS if c in feats]
    levels = {c: sorted(ref[c].astype(str).unique()) for c in cats}
    def conv(df):
        X = df[feats].copy()
        for c in cats:
            X[c] = pd.Categorical(X[c].astype(str), categories=levels[c])
        return X
    return conv

def build_matrices(feats, fit_df, dfs):
    """Quatre représentations des mêmes variables, ajustées sur fit_df :
    lin = numériques plafonnées + standardisées + one-hot (modèles linéaires, voisins, SVM, réseaux)
    ord = numériques + catégories encodées en entiers (arbres, forêts, boosting sklearn et XGBoost)
    lgb = DataFrame avec dtype category (LightGBM)
    cat = DataFrame avec catégories en texte (CatBoost)"""
    num = [c for c in feats if c not in CATS]
    cats = [c for c in feats if c in CATS]
    lin = ColumnTransformer([("num", Pipeline([("clip", Clipper()), ("sc", StandardScaler())]), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cats)],
                            sparse_threshold=0).fit(fit_df[feats])
    ordt = ColumnTransformer([("num", "passthrough", num),
                              ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cats)]).fit(fit_df[feats])
    conv = make_converter(feats, fit_df)
    M = {"lin": {}, "ord": {}, "lgb": {}, "cat": {}}
    for n, d in dfs.items():
        M["lin"][n] = np.asarray(lin.transform(d[feats]), dtype=float)
        M["ord"][n] = np.asarray(ordt.transform(d[feats]), dtype=float)
        M["lgb"][n] = conv(d)
        X = d[feats].copy()
        for c in cats:
            X[c] = X[c].astype(str)
        M["cat"][n] = X
    return M

print("Fonctions de préparation prêtes.")

## 5. Registre des modèles de classification
Le déséquilibre est géré par `class_weight` (modèles sklearn), `scale_pos_weight` (XGBoost, LightGBM) ou `auto_class_weights` (CatBoost). Les modèles sans probabilités (SVM, Ridge) fournissent un score de décision : suffisant pour PR-AUC et ROC-AUC.

In [ ]:
SPW = float(np.sqrt(RATIO))
LGB_BASE = dict(n_estimators=N_EST, learning_rate=0.05, num_leaves=31, min_child_samples=50, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, scale_pos_weight=SPW,
                random_state=SEED, n_jobs=-1, verbose=-1)
XGB_BASE = dict(n_estimators=N_EST, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                min_child_weight=5, scale_pos_weight=SPW, tree_method="hist", random_state=SEED, n_jobs=-1,
                eval_metric="aucpr")
CAT_BASE = dict(iterations=N_EST, learning_rate=0.08, depth=6, random_seed=SEED, verbose=0, thread_count=-1,
                auto_class_weights="SqrtBalanced")

ne = [100, 150] if FAST else [200, 400, 700]
spw_grid = [1.0, SPW, RATIO]
GRID_LGB = {"num_leaves": [15, 31, 63], "learning_rate": [0.02, 0.03, 0.05, 0.08], "min_child_samples": [20, 50, 100, 200],
            "subsample": [0.7, 0.8, 0.9, 1.0], "colsample_bytree": [0.5, 0.7, 0.9], "reg_lambda": [0, 1, 5, 20],
            "n_estimators": ne, "scale_pos_weight": spw_grid}
GRID_XGB = {"max_depth": [3, 4, 6], "learning_rate": [0.02, 0.03, 0.05, 0.08], "min_child_weight": [1, 5, 20, 50],
            "subsample": [0.7, 0.8, 0.9, 1.0], "colsample_bytree": [0.5, 0.7, 0.9], "reg_lambda": [0.5, 1, 5, 20],
            "n_estimators": ne, "scale_pos_weight": spw_grid}
GRID_CAT = {"depth": [4, 6, 8], "learning_rate": [0.03, 0.05, 0.08, 0.12], "l2_leaf_reg": [1, 3, 10, 30], "iterations": ne}

def build_registry():
    R = {}
    def add(name, family, prep, make, sub=None, grid=None):
        R[name] = {"family": family, "prep": prep, "make": make, "sub": sub, "grid": grid}
    add("Régression logistique", "Linéaire", "lin", lambda p=None: LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
    add("Ridge classifier", "Linéaire", "lin", lambda p=None: RidgeClassifier(alpha=10.0, class_weight="balanced"))
    add("LDA", "Discriminant", "lin", lambda p=None: LinearDiscriminantAnalysis())
    add("Naive Bayes (gaussien)", "Bayésien", "lin", lambda p=None: GaussianNB())
    add("KNN", "Voisins", "lin", lambda p=None: KNeighborsClassifier(n_neighbors=50, weights="distance", n_jobs=-1))
    add("SVM (rbf)", "Noyau", "lin", lambda p=None: SVC(C=1.0, kernel="rbf", class_weight="balanced"), sub=SLOW_SUB)
    add("SVM linéaire", "Noyau", "lin", lambda p=None: LinearSVC(C=0.1, class_weight="balanced", max_iter=5000, random_state=SEED))
    add("MLP", "Réseau de neurones", "lin", lambda p=None: MLPClassifier(hidden_layer_sizes=(128, 64), early_stopping=True,
        max_iter=60 if FAST else 200, random_state=SEED))
    add("Decision Tree", "Arbre", "ord", lambda p=None: DecisionTreeClassifier(max_depth=8, min_samples_leaf=100, class_weight="balanced", random_state=SEED))
    add("Random Forest", "Bagging", "ord", lambda p=None: RandomForestClassifier(n_estimators=RF_TREES, max_depth=14,
        min_samples_leaf=30, max_features=0.5, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))
    add("Extra Trees", "Bagging", "ord", lambda p=None: ExtraTreesClassifier(n_estimators=RF_TREES, max_depth=16,
        min_samples_leaf=30, max_features=0.7, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))
    add("Gradient Boosting (sklearn)", "Boosting", "ord", lambda p=None: GradientBoostingClassifier(n_estimators=GB_TREES,
        learning_rate=0.08, max_depth=3, subsample=0.8, random_state=SEED))
    add("HistGradientBoosting", "Boosting", "ord", lambda p=None: HistGradientBoostingClassifier(max_iter=2 * GB_TREES,
        learning_rate=0.06, random_state=SEED))
    add("AdaBoost", "Boosting", "ord", lambda p=None: AdaBoostClassifier(DecisionTreeClassifier(max_depth=3),
        n_estimators=60 if FAST else 100, learning_rate=0.5, random_state=SEED))
    add("LightGBM", "Boosting moderne", "lgb", lambda p=None: lgb.LGBMClassifier(**{**LGB_BASE, **(p or {})}), grid=GRID_LGB)
    if HAS_XGB:
        add("XGBoost", "Boosting moderne", "ord", lambda p=None: xgb.XGBClassifier(**{**XGB_BASE, **(p or {})}), grid=GRID_XGB)
    if HAS_CAT:
        add("CatBoost", "Boosting moderne", "cat", lambda p=None: CatBoostClassifier(**{**CAT_BASE, **(p or {})}), grid=GRID_CAT)
    return {k: v for k, v in R.items() if k not in SKIP_MODELS}

CLF = build_registry()
print(len(CLF), "modèles :", ", ".join(CLF))

def fit_one(spec, M, y, params=None, rows=None):
    X, yy = M[spec["prep"]]["fit"], y
    if rows is not None:
        X, yy = take(X, rows), y[rows]
    if spec["sub"] and len(yy) > spec["sub"]:
        idx = np.sort(rng.choice(len(yy), spec["sub"], replace=False))
        X, yy = take(X, idx), yy[idx]
    est = spec["make"](params)
    if spec["prep"] == "cat":
        est.fit(X, yy, cat_features=[c for c in CATS if c in X.columns])
    else:
        est.fit(X, yy)
    return est

def score_of(est, X):
    """Probabilité de retard si disponible, sinon score de décision."""
    if hasattr(est, "predict_proba"):
        return np.asarray(est.predict_proba(X))[:, 1]
    return np.asarray(est.decision_function(X)).ravel()

## 6. Faut-il utiliser la promesse d'Olist comme variable ?
Comparaison sur valid de deux modèles représentatifs (Régression logistique, LightGBM), **avec et sans** `promised_days`, selon la PR-AUC.

In [ ]:
sets = {"sans promesse": FEATURES_CORE, "avec promesse": FEATURES_CORE + PROMISE}
probe = []
for set_name, feats in sets.items():
    Mp = build_matrices(feats, tr, {"fit": tr, "va": va})
    for name in ["Régression logistique", "LightGBM"]:
        spec = CLF[name]
        est = fit_one(spec, Mp, ytr)
        probe.append({"modèle": name, "variables": set_name, **clf_metrics(yva, score_of(est, Mp[spec["prep"]]["va"]))})
probe = pd.DataFrame(probe)
display(probe[["modèle", "variables", "PR-AUC", "ROC-AUC", "F1 max"]])

if USE_PROMISE is None:
    use_promise = probe[probe["modèle"] == "LightGBM"].sort_values("PR-AUC", ascending=False).iloc[0]["variables"] == "avec promesse"
else:
    use_promise = bool(USE_PROMISE)
FEATS = FEATURES_CORE + (PROMISE if use_promise else [])
print("Variables retenues :", "avec" if use_promise else "sans", "promesse")

## 7. Entraînement et comparaison de tous les modèles (valid)
Chaque modèle est entraîné sur le **train** et évalué sur **valid**. Colonnes : PR-AUC (métrique de classement principale), ROC-AUC, F1 maximal, précision et rappel parmi les 10 % de commandes les plus risquées, et temps d'entraînement.

In [ ]:
M = build_matrices(FEATS, tr, {"fit": tr, "va": va, "te": te})
fitted, s_va, rows = {}, {}, []
for name, spec in CLF.items():
    t0 = time.time()
    try:
        est = fit_one(spec, M, ytr)
        s = score_of(est, M[spec["prep"]]["va"])
        fitted[name], s_va[name] = est, s
        rows.append({"modèle": name, "famille": spec["family"], **clf_metrics(yva, s, proba=hasattr(est, "predict_proba")),
                     "temps (s)": time.time() - t0})
        print(f"  {name:30s} PR-AUC valid = {rows[-1]['PR-AUC']:.3f}   ({rows[-1]['temps (s)']:.0f} s)")
    except Exception as e:
        print(f"  {name:30s} ÉCHEC : {type(e).__name__}: {str(e)[:80]}")

comparison = pd.DataFrame(base_rows + rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
comparison.insert(0, "rang", comparison.index + 1)
comparison["gain vs taux de base"] = comparison["PR-AUC"] / float(yva.mean())
display(comparison)

## 8. Visualisation de la comparaison

In [ ]:
plot_df = comparison.sort_values("PR-AUC")
colors = ["tab:red" if f == "Baseline" else "tab:blue" for f in plot_df["famille"]]
fig, ax = plt.subplots(1, 3, figsize=(24, 0.38 * len(plot_df) + 2))
ax[0].barh(plot_df["modèle"], plot_df["PR-AUC"], color=colors); ax[0].axvline(yva.mean(), color="k", ls=":")
ax[0].set_title("PR-AUC sur valid (pointillés : taux de base ; rouge : baselines)")
mdl = comparison[comparison["famille"] != "Baseline"]
ax[1].scatter(mdl["ROC-AUC"], mdl["PR-AUC"], s=60)
for _, r in mdl.iterrows():
    ax[1].annotate(r["modèle"], (r["ROC-AUC"], r["PR-AUC"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax[1].set_xlabel("ROC-AUC"); ax[1].set_ylabel("PR-AUC"); ax[1].set_title("ROC-AUC contre PR-AUC")
mdl.groupby("famille")["PR-AUC"].max().sort_values().plot.barh(ax=ax[2], color="tab:green")
ax[2].set_title("Meilleure PR-AUC de chaque famille")
plt.tight_layout(); plt.show()

## 9. Réglage des meilleurs boosters (validation croisée temporelle, PR-AUC)
Les **3 meilleurs modèles disposant d'une grille** (LightGBM, XGBoost, CatBoost) sont réglés avec `TimeSeriesSplit` sur le train.

In [ ]:
tunable = [n for n in comparison["modèle"] if n in CLF and CLF[n]["grid"] is not None and n in fitted][:3]
print("Modèles à régler :", tunable)
tscv = TimeSeriesSplit(n_splits=3)
best_params, tuned_est, tuned_va, log = {}, {}, {}, []
for name in tunable:
    spec = CLF[name]; results = []
    for params in ParameterSampler(spec["grid"], n_iter=N_ITER_SEARCH, random_state=SEED):
        sc = []
        for a, b in tscv.split(np.arange(len(tr))):
            if ytr[a].sum() == 0 or ytr[b].sum() == 0:
                continue
            est = fit_one(spec, M, ytr, params, rows=a)
            sc.append(pr_auc(ytr[b], score_of(est, take(M[spec["prep"]]["fit"], b))))
        results.append((float(np.mean(sc)) if sc else -1.0, params))
    results.sort(key=lambda r: r[0], reverse=True)
    best_params[name] = results[0][1]
    est = fit_one(spec, M, ytr, results[0][1])
    tuned_est[name] = est
    tuned_va[name] = score_of(est, M[spec["prep"]]["va"])
    log.append({"modèle": name, "PR-AUC cv": results[0][0],
                "PR-AUC valid défaut": comparison.loc[comparison["modèle"] == name, "PR-AUC"].iloc[0],
                "PR-AUC valid réglé": pr_auc(yva, tuned_va[name]), "paramètres": results[0][1]})
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame(log))

## 10. Mélange des modèles réglés et choix de la configuration finale

In [ ]:
blend_va = np.mean([tuned_va[n] for n in tunable], axis=0)
cand = {n + " (réglé)": tuned_va[n] for n in tunable}
cand["Mélange (moyenne des probabilités)"] = blend_va
final_valid = pd.DataFrame([{"modèle": k, **clf_metrics(yva, v)} for k, v in cand.items()]).sort_values("PR-AUC", ascending=False)
display(final_valid.reset_index(drop=True))
best_final = final_valid.iloc[0]["modèle"]
print("Meilleure configuration sur valid :", best_final)

## 11. Probabilités sur valid et test
La configuration retenue, **entraînée sur le train seul**, fournit les scores de valid (pour choisir les seuils) puis de test (pour les évaluer) : procédure cohérente. Les finalistes sont évalués sur test **une seule fois**.

In [ ]:
tuned_te = {n: score_of(tuned_est[n], M[CLF[n]["prep"]]["te"]) for n in tunable}
blend_te = np.mean([tuned_te[n] for n in tunable], axis=0)
if best_final.startswith("Mélange"):
    p_va, p_te, components = blend_va, blend_te, tunable
else:
    nm = best_final.replace(" (réglé)", "")
    p_va, p_te, components = tuned_va[nm], tuned_te[nm], [nm]

rows = [{"modèle": "Taux de base", "PR-AUC": float(yte.mean()), "ROC-AUC": 0.5}]
rows += [{"modèle": n + " (réglé)", **clf_metrics(yte, tuned_te[n])} for n in tunable]
rows.append({"modèle": "Mélange (moyenne des probabilités)", **clf_metrics(yte, blend_te)})
test_res = pd.DataFrame(rows).set_index("modèle")
test_res["gain vs taux de base"] = test_res["PR-AUC"] / float(yte.mean())
display(test_res)
print(f"Configuration retenue sur valid : {best_final}")

## 12. Choix du seuil de décision (valid) et évaluation (test)
Trois politiques : **coût minimal** (`COST_FN` × retards manqués + `COST_FP` × fausses alertes), **précision cible**, **capacité** (top `TOPK_SHARE`). Comparées à ne rien signaler et tout signaler.

In [ ]:
def eval_at(y, flag):
    flag, y = np.asarray(flag).astype(bool), np.asarray(y).astype(bool)
    tp, fp, fn = int((flag & y).sum()), int((flag & ~y).sum()), int((~flag & y).sum())
    return {"signalées (%)": flag.mean() * 100, "précision": tp / max(tp + fp, 1), "rappel": tp / max(tp + fn, 1),
            "F1": 2 * tp / max(2 * tp + fp + fn, 1), "retards manqués": fn, "fausses alertes": fp,
            "coût": COST_FN * fn + COST_FP * fp}

def thr_cost(y, s):
    grid = np.unique(np.quantile(s, np.linspace(0, 1, 501)))
    return min((COST_FN * ((s < t) & (y == 1)).sum() + COST_FP * ((s >= t) & (y == 0)).sum(), t) for t in grid)[1]

def thr_precision(y, s, target):
    prec, rec, thr = precision_recall_curve(y, s)
    ok = np.where(prec[:-1] >= target)[0]
    return thr[ok[np.argmax(rec[:-1][ok])]] if len(ok) else np.inf

def thr_topk(s, share): return float(np.quantile(s, 1 - share))

thresholds = {"Coût minimal": thr_cost(yva, p_va), "Précision cible": thr_precision(yva, p_va, PRECISION_TARGET),
              "Capacité (top %d %%)" % round(TOPK_SHARE * 100): thr_topk(p_va, TOPK_SHARE)}
rows = []
for name, t in thresholds.items():
    rows.append({"politique": name, "seuil": t, "bloc": "valid", **eval_at(yva, p_va >= t)})
    rows.append({"politique": name, "seuil": t, "bloc": "test", **eval_at(yte, p_te >= t)})
for name, f in [("Ne rien signaler", np.zeros(len(yte))), ("Tout signaler", np.ones(len(yte)))]:
    rows.append({"politique": name, "seuil": np.nan, "bloc": "test", **eval_at(yte, f)})
policy = pd.DataFrame(rows)
display(policy[policy["bloc"] == "test"].set_index("politique").drop(columns="bloc"))
print("Lecture : le taux de retard de valid (13,5 %) diffère de celui de test (6,2 %) ; un seuil choisi sur valid peut signaler plus de commandes sur test.")
display(policy[policy["bloc"] == "valid"].set_index("politique")[["seuil", "signalées (%)", "précision", "rappel"]])

## 13. Courbes de performance : précision-rappel, gain cumulé, calibration

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for name, y, s, col in [("valid", yva, p_va, "tab:blue"), ("test", yte, p_te, "tab:orange")]:
    prec, rec, _ = precision_recall_curve(y, s)
    ax[0].plot(rec, prec, color=col, label=f"{name} (PR-AUC {pr_auc(y, s):.3f})"); ax[0].axhline(y.mean(), color=col, ls=":", alpha=0.6)
ax[0].set_xlabel("Rappel"); ax[0].set_ylabel("Précision"); ax[0].set_title("Précision-rappel (pointillés : taux de base)"); ax[0].legend()

order = np.argsort(-p_te)
ax[1].plot(np.arange(1, len(yte) + 1) / len(yte), np.cumsum(yte[order]) / max(yte.sum(), 1), label="Modèle")
ax[1].plot([0, 1], [0, 1], "k--", label="Aléatoire"); ax[1].axvline(TOPK_SHARE, color="red", ls=":", label=f"Top {TOPK_SHARE*100:.0f} %")
ax[1].set_xlabel("Part des commandes signalées"); ax[1].set_ylabel("Part des retards capturés"); ax[1].set_title("Gain cumulé (test)"); ax[1].legend()

for name, y, s, col in [("valid", yva, p_va, "tab:blue"), ("test", yte, p_te, "tab:orange")]:
    if 0 <= s.min() and s.max() <= 1:
        fr, mp = calibration_curve(y, s, n_bins=10, strategy="quantile"); ax[2].plot(mp, fr, "o-", color=col, label=name)
ax[2].plot([0, 1], [0, 1], "k--"); ax[2].set_xlabel("Probabilité prédite"); ax[2].set_ylabel("Fréquence observée")
ax[2].set_title("Calibration"); ax[2].legend()
plt.tight_layout(); plt.show()

pt, rt = top_stats(yte, p_te, TOPK_SHARE)
print(f"Top {TOPK_SHARE*100:.0f} % des commandes (test) : précision {pt*100:.1f} % (taux de base {yte.mean()*100:.1f} %, "
      f"gain x{pt/max(yte.mean(),1e-9):.1f}), rappel {rt*100:.1f} %")

## 14. Mini-expérience : classification directe contre « régression puis seuil »
Un régresseur LightGBM (objectif L1) prédit la durée ; le **score de risque** est l'écart `durée prédite - promesse`. Comparaison par PR-AUC et à capacité égale (top 10 %).

In [ ]:
if HAVE_M1:
    conv_r = make_converter(FEATS, tr)
    reg = lgb.LGBMRegressor(**{k: v for k, v in LGB_BASE.items() if k != "scale_pos_weight"}, objective="regression_l1").fit(
        conv_r(tr), tr["delivery_days"].values)
    gap_va = reg.predict(conv_r(va)) - va["promised_days"].values
    gap_te = reg.predict(conv_r(te)) - te["promised_days"].values
    rows = []
    for approche, a_va, a_te in [("Classification directe", p_va, p_te), ("Régression puis seuil (durée - promesse)", gap_va, gap_te)]:
        ev_cost, ev_top = eval_at(yte, a_te >= thr_cost(yva, a_va)), eval_at(yte, a_te >= thr_topk(a_va, TOPK_SHARE))
        rows.append({"approche": approche, "PR-AUC valid": pr_auc(yva, a_va), "PR-AUC test": pr_auc(yte, a_te),
                     "ROC-AUC test": roc_auc(yte, a_te), "coût test (seuil coût)": ev_cost["coût"],
                     "rappel (seuil coût)": ev_cost["rappel"], "précision (seuil coût)": ev_cost["précision"],
                     "rappel top 10 %": ev_top["rappel"], "précision top 10 %": ev_top["précision"]})
    exp = pd.DataFrame(rows).set_index("approche"); display(exp.T)
    print(f"Coût de référence sur test sans aucune alerte : {COST_FN * int(yte.sum()):.0f}")
else:
    exp = pd.DataFrame(); print("Mini-expérience ignorée (dataset du modèle 1 absent).")

## 15. Facteurs de risque (LightGBM, contributions de type SHAP)

In [ ]:
lgb_p = {**LGB_BASE, **best_params.get("LightGBM", {})}
lgb_interp = lgb.LGBMClassifier(**lgb_p).fit(M["lgb"]["fit"], ytr)
sample = M["lgb"]["te"].sample(min(3000, len(te)), random_state=SEED)
contrib = lgb_interp.booster_.predict(sample, pred_contrib=True)[:, :-1]
imp = pd.Series(np.abs(contrib).mean(axis=0), index=sample.columns).sort_values(ascending=False)
imp.head(20)[::-1].plot.barh(figsize=(8, 7), title="Contribution moyenne absolue (log-cotes)"); plt.show()
display(imp.head(15).to_frame("contribution moyenne absolue"))
try:
    import shap
    codes = sample.copy()
    for c in [c for c in CATS if c in codes.columns]:
        codes[c] = codes[c].cat.codes
    shap.summary_plot(contrib, codes, max_display=15, show=True)
except Exception as e:
    print("Graphique SHAP non affiché :", type(e).__name__, e)

## 16. Points faibles : rappel et précision par mois et par état (politique coût minimal)

In [ ]:
t = thresholds["Coût minimal"]
diag = te[["customer_state", "order_purchase_timestamp"]].copy()
diag["y"], diag["flag"] = yte, (p_te >= t).astype(int)
diag["mois"] = diag["order_purchase_timestamp"].dt.to_period("M").astype(str)

def summarize(g):
    tp = int(((g.flag == 1) & (g.y == 1)).sum())
    return pd.Series({"commandes": len(g), "taux_retard": g.y.mean(), "signalées": g.flag.mean(),
                      "rappel": tp / max(g.y.sum(), 1), "précision": tp / max(g.flag.sum(), 1)})
print("Par mois :"); display(diag.groupby("mois").apply(summarize))
top_states = diag["customer_state"].value_counts().head(10).index
print("Par état (10 plus fréquents) :")
display(diag[diag["customer_state"].isin(top_states)].groupby("customer_state").apply(summarize).sort_values("taux_retard", ascending=False))

## 17. Sauvegarde

In [ ]:
trva = pd.concat([tr, va], ignore_index=True)
M2 = build_matrices(FEATS, trva, {"fit": trva, "te": te})
refit = {n: fit_one(CLF[n], M2, trva["is_late"].values, best_params[n]) for n in components}
joblib.dump(refit, os.path.join(OUT_DIR, "model2_final_components.joblib"))

pred = pd.concat([va[["order_id", "split", "is_late"]].assign(score_retard=p_va),
                  te[["order_id", "split", "is_late"]].assign(score_retard=p_te)])
pred.to_csv(os.path.join(OUT_DIR, "model2_predictions.csv"), index=False)
comparison.to_csv(os.path.join(OUT_DIR, "model2_all_models_valid.csv"), index=False)

summary = {"features": FEATS, "use_promise": bool(use_promise), "final": best_final, "components": components,
           "costs": {"COST_FN": COST_FN, "COST_FP": COST_FP},
           "thresholds_from_valid": {k: (None if not np.isfinite(v) else float(v)) for k, v in thresholds.items()},
           "metrics": {"valid": {"PR_AUC": pr_auc(yva, p_va), "base_rate": float(yva.mean())},
                       "test": {"PR_AUC": pr_auc(yte, p_te), "base_rate": float(yte.mean())}},
           "policies_test": policy[policy["bloc"] == "test"].drop(columns="bloc").to_dict("records"),
           "mini_experiment": exp.reset_index().to_dict("records") if len(exp) else []}
with open(os.path.join(OUT_DIR, "model2_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=float)
print("Fichiers :", sorted(os.listdir(OUT_DIR)))

## Synthèse du modèle 2
À compléter avec tes résultats :
- **Classement des familles** : PR-AUC de chaque modèle contre le taux de base et les règles simples (cellules 7 et 8).
- **Promesse d'Olist comme variable** : comparaison avec et sans (cellule 6).
- **Réglage et mélange** : gain par rapport aux modèles par défaut (cellules 9 à 11).
- **Politique de décision** : coût, précision et rappel de chaque seuil sur test, et effet de la dérive du taux de retard (cellule 12).
- **Classification contre régression + seuil** : résultat de la mini-expérience (cellule 14).
- **Limites** : retards souvent dus à des événements non observés (transporteur, météo, grèves) ; taux de retard instable.

Étape suivante : `04c-model3-customer-segmentation`.